# ch8 — Tiny ImageNet-200 训练 (Kaggle)

支持 VGG11 / VGG13 / VGG16 / VGG19，自动检测 CUDA / 昇腾 NPU / CPU。

> 需要在 Kaggle Dataset 中手动添加 [tiny-imagenet-200-zip](https://www.kaggle.com/datasets/wwhds/tiny-imagenet-200-zip) 。

In [ ]:
# Cell 1: 安装依赖（Kaggle 首次运行时需要）
!pip install torch torchvision Pillow tqdm matplotlib -q
print("依赖安装完成")

In [ ]:
# Cell 2: 导入依赖
import os
import sys
import json
import time
import random
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print("导入完成")

In [ ]:
# Cell 3: 训练配置 — 所有参数集中在这里修改
# ═══════════════════════════════════════════════════
CONFIG = {
    # 模型
    "model_name": "VGG11",        # VGG11 / VGG13 / VGG16 / VGG19
    "num_classes": 200,

    # 训练超参
    "epochs": 15,
    "batch_size": 128,
    "lr": 0.01,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "dropout": 0.5,

    # 早停（0 = 不早停）
    "early_stop_patience": 0,

    # 设备（auto / cuda / npu / cpu）
    "device": "auto",
}
# ═══════════════════════════════════════════════════

print("当前配置:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 4: Kaggle 数据集路径设置
# Kaggle 上需要手动添加 tiny-imagenet-200-zip 数据集

# 优先检查已解压的文件夹
DATA_PARENT = Path("/kaggle/input/tiny-imagenet-200-zip")
data_dir = DATA_PARENT / "tiny-imagenet-200"

# 如果只有 zip，则解压到 working 目录
zip_path = DATA_PARENT / "tiny-imagenet-200.zip"
if not data_dir.exists() and zip_path.exists():
    print("正在解压 tiny-imagenet-200.zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/kaggle/working/")
    data_dir = Path("/kaggle/working/tiny-imagenet-200")
    print(f"已解压到: {data_dir}")

if not data_dir.exists():
    raise FileNotFoundError(
        f"数据集未找到。请确认已添加 tiny-imagenet-200-zip 数据集到当前 notebook。"
    )

# TinyImageNet 类需要父目录作为 root
DATA_ROOT = str(data_dir.parent)
print(f"数据集路径: {data_dir}")
print(f"DataLoader root: {DATA_ROOT}")

In [ ]:
# Cell 5: Tiny ImageNet-200 数据集

class TinyImageNet(Dataset):
    """Tiny ImageNet-200 数据集，支持 train / val 两种 split。"""

    def __init__(self, root, split="train", transform=None):
        self.root = Path(root) / "tiny-imagenet-200"
        self.transform = transform
        self._build_label_map()

        if split == "train":
            self._load_train()
        elif split == "val":
            self._load_val()
        else:
            raise ValueError(f"split must be 'train' or 'val', got {split!r}")

    def _build_label_map(self):
        wnids_path = self.root / "wnids.txt"
        wnids = wnids_path.read_text().strip().splitlines()
        self._wnid2label = {w: i for i, w in enumerate(wnids)}

    def _load_train(self):
        self.images, self.labels = [], []
        for cls_dir in sorted(self.root.glob("train/*")):
            wnid = cls_dir.name
            label = self._wnid2label[wnid]
            img_dir = cls_dir / "images"
            if img_dir.is_dir():
                for img_path in sorted(img_dir.glob("*.JPEG")):
                    self.images.append(img_path)
                    self.labels.append(label)

    def _load_val(self):
        self.images, self.labels = [], []
        annot_path = self.root / "val/val_annotations.txt"
        for line in annot_path.read_text().strip().splitlines():
            parts = line.split()
            img_path = self.root / "val/images" / parts[0]
            wnid = parts[1]
            self.images.append(img_path)
            self.labels.append(self._wnid2label[wnid])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

    @property
    def class_names(self):
        if hasattr(self, "_class_names"):
            return self._class_names
        words_path = self.root / "words.txt"
        wnid2word = {}
        for line in words_path.read_text().strip().splitlines():
            parts = line.split("\t")
            if len(parts) == 2:
                wnid2word[parts[0]] = parts[1].split(",")[0].strip()
        self._class_names = [
            wnid2word.get(w, w)
            for w in self.root.joinpath("wnids.txt").read_text().strip().splitlines()
        ]
        return self._class_names


def get_transform(train=True):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
    if train:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(64, padding=4),
            transforms.ToTensor(),
            normalize,
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])


def get_loaders(data_root, batch_size=128, num_workers=2):
    train_set = TinyImageNet(data_root, split="train",
                             transform=get_transform(train=True))
    val_set = TinyImageNet(data_root, split="val",
                           transform=get_transform(train=False))
    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers,
    )
    return train_loader, val_loader


print("数据集代码加载完成")

In [ ]:
# Cell 6: VGG 模型 + 模型注册表
# 添加新模型时：1) 定义模型类；2) 注册到 MODEL_REGISTRY

VGG_CFGS = {
    "VGG11":  [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "VGG13":  [64, 64, "M", 128, 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "VGG16":  [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"],
    "VGG19":  [64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M", 512, 512, 512, 512, "M", 512, 512, 512, 512, "M"],
}


class VGG(nn.Module):
    """通用 VGG 实现。"""

    def __init__(self, name="VGG16", in_channels=3, num_classes=200,
                 fc_channels=None, dropout=0.5):
        super().__init__()
        cfg = VGG_CFGS[name]
        fc_channels = fc_channels or [4096, 4096]
        self.features = self._make_layers(cfg, in_channels)

        pool_times = sum(1 for v in cfg if v == "M")
        final_size = 64 // (2 ** pool_times)
        fc_input = 512 * final_size * final_size

        fc_layers = []
        for ch in fc_channels:
            fc_layers.extend([
                nn.Linear(fc_input, ch),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ])
            fc_input = ch
        fc_layers.append(nn.Linear(fc_input, num_classes))
        self.classifier = nn.Sequential(*fc_layers)
        self._init_weights()

    def _make_layers(self, cfg, in_channels):
        layers = []
        for v in cfg:
            if v == "M":
                layers.append(nn.MaxPool2d(2, 2))
            else:
                layers.append(nn.Conv2d(in_channels, v, 3, padding=1))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# 模型注册表 — 添加新模型只需在此注册
MODEL_REGISTRY = {
    "VGG11": lambda **kw: VGG("VGG11", **kw),
    "VGG13": lambda **kw: VGG("VGG13", **kw),
    "VGG16": lambda **kw: VGG("VGG16", **kw),
    "VGG19": lambda **kw: VGG("VGG19", **kw),
}


def get_model(name, **kwargs):
    if name not in MODEL_REGISTRY:
        available = ", ".join(MODEL_REGISTRY)
        raise KeyError(f"未知模型 {name!r}，可选: {available}")
    return MODEL_REGISTRY[name](**kwargs)


print(f"模型注册表已就绪，可用模型: {list(MODEL_REGISTRY.keys())}")

### 添加新模型

如需添加新模型（如 ResNet），在上方 Cell 6 末尾插入新代码 cell：

```python
class ResNet18(nn.Module):
    ...

MODEL_REGISTRY["ResNet18"] = lambda **kw: ResNet18(**kw)
```

然后回到 Cell 3 修改 `model_name = "ResNet18"` 即可。

In [ ]:
# Cell 7: 设备检测
device_str = CONFIG["device"]
if device_str == "auto":
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch, "npu") and torch.npu.is_available():
        device = torch.device("npu")
    else:
        device = torch.device("cpu")
else:
    device = torch.device(device_str)

print(f"设备: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif device.type == "npu":
    print(f"NPU: {torch.npu.get_device_name(0)}")

In [ ]:
# Cell 8: 加载数据（从 Kaggle Dataset）
print("正在加载 Tiny ImageNet-200...")
train_loader, val_loader = get_loaders(
    DATA_ROOT,
    CONFIG["batch_size"],
)
print(f"训练集: {len(train_loader.dataset)} 张")
print(f"验证集: {len(val_loader.dataset)} 张")
print(f"每 epoch 迭代数: {len(train_loader)}")

In [ ]:
# Cell 9: 创建模型 + 打印参数量
model = get_model(
    CONFIG["model_name"],
    num_classes=CONFIG["num_classes"],
    dropout=CONFIG["dropout"],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型: {CONFIG['model_name']}")
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print(f"模型大小: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

In [ ]:
# Cell 10: 训练组件
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=CONFIG["lr"],
    momentum=CONFIG["momentum"],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.1, patience=5
)

print("训练组件已就绪")

In [ ]:
# Cell 11: 训练循环

epochs = CONFIG["epochs"]
patience = CONFIG["early_stop_patience"]

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [],   "val_acc": [],
    "epoch_time": [],
}
best_acc = 0
best_epoch = 0
wait = 0

total_start = time.time()

print(f"{'Epoch':>5}  {'Train Loss':>11}  {'Train Acc':>10}  "
      f"{'Val Loss':>9}  {'Val Acc':>9}  {'Time':>6}")
print("-" * 60)

for epoch in range(1, epochs + 1):
    epoch_start = time.time()

    # ── 训练 ──
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    pbar = tqdm(train_loader, desc=f"  Epoch {epoch}/{epochs}",
                leave=False, ncols=80)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        batch_size = inputs.size(0)
        train_total += batch_size

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(targets).sum().item()

        pbar.set_postfix(loss=loss.item(),
                         acc=f"{100. * train_correct / train_total:.1f}%")

    train_loss /= len(train_loader)
    train_acc = 100. * train_correct / train_total

    # ── 验证 ──
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc="  val",
                                    leave=False, ncols=80):
            inputs, targets = inputs.to(device), targets.to(device)
            val_total += inputs.size(0)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(targets).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100. * val_correct / val_total

    epoch_time = time.time() - epoch_start

    # ── 记录 ──
    history["train_loss"].append(round(train_loss, 4))
    history["train_acc"].append(round(train_acc, 2))
    history["val_loss"].append(round(val_loss, 4))
    history["val_acc"].append(round(val_acc, 2))
    history["epoch_time"].append(round(epoch_time, 1))

    scheduler.step(val_acc)

    marker = " *" if val_acc > best_acc else ""
    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch
        wait = 0
    else:
        wait += 1

    print(f"{epoch:5d}  {train_loss:11.4f}  {train_acc:9.2f}%  "
          f"{val_loss:9.4f}  {val_acc:8.2f}%  {epoch_time:5.0f}s{marker}")

    # 早停检查
    if patience > 0 and wait >= patience:
        print(f"\n早停: {patience} 个 epoch 无改善，停止训练")
        break

total_time = time.time() - total_start
print(f"\n总训练时长: {total_time:.0f}s ({total_time/60:.1f}min)")
print(f"最佳验证准确率: {best_acc:.2f}% (epoch {best_epoch})")
print(f"总 epoch 数: {epoch}")

In [ ]:
# Cell 12: 结果可视化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 左图: Accuracy 对比 ──
ax = axes[0]
epochs_range = range(1, len(history["train_acc"]) + 1)
ax.plot(epochs_range, history["train_acc"], "b-o", label="Train Acc", markersize=4)
ax.plot(epochs_range, history["val_acc"], "r-s", label="Val Acc", markersize=4)
ax.axvline(x=best_epoch, color="gray", linestyle="--", alpha=0.5,
           label=f"Best (epoch {best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Train vs Validation Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# ── 右图: Loss 对比 ──
ax = axes[1]
ax.plot(epochs_range, history["train_loss"], "b-o", label="Train Loss", markersize=4)
ax.plot(epochs_range, history["val_loss"], "r-s", label="Val Loss", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Train vs Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印 summary
print(f"""
═══ 训练总结 ═══
模型:         {CONFIG['model_name']}
设备:         {device}
总时长:       {total_time:.0f}s ({total_time/60:.1f}min)
最佳 Val Acc: {best_acc:.2f}% (epoch {best_epoch})
最终 Val Acc: {history['val_acc'][-1]:.2f}%
最终 Train Acc: {history['train_acc'][-1]:.2f}%
Train-Val 差距: {history['train_acc'][-1] - history['val_acc'][-1]:.1f}%
""")

In [ ]:
# Cell 13: 预测样例展示 — 随机抽验证集图片看模型表现

val_set = val_loader.dataset
class_names = val_set.class_names

# 随机抽 6 张
indices = random.sample(range(len(val_set)), 6)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
model.eval()

with torch.no_grad():
    for ax, idx in zip(axes.flatten(), indices):
        img_tensor, true_label = val_set[idx]
        # 反标准化以显示
        img = img_tensor.cpu().clone()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img * std + mean
        img = img.permute(1, 2, 0).numpy()
        img = img.clip(0, 1)

        # 推理
        output = model(img_tensor.unsqueeze(0).to(device))
        pred_label = output.argmax(1).item()

        correct = pred_label == true_label
        color = "green" if correct else "red"
        true_name = class_names[true_label] if true_label < len(class_names) else str(true_label)
        pred_name = class_names[pred_label] if pred_label < len(class_names) else str(pred_label)

        ax.imshow(img)
        ax.set_title(f"True: {true_name}\nPred: {pred_name}",
                     fontsize=9, color=color)
        ax.axis("off")

plt.suptitle(f"{CONFIG['model_name']} — 验证集预测样例 (绿色=正确, 红色=错误)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: 保存产物
artifact_dir = Path("./artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / f"{CONFIG['model_name']}_tiny_imagenet.pth"
torch.save(model.state_dict(), model_path)

history_path = artifact_dir / f"{CONFIG['model_name']}_tiny_imagenet_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print(f"模型已保存: {model_path}")
print(f"训练记录已保存: {history_path}")
print("\n训练全部完成!")